# nb_editing_tools

> MCP tools for reading, editing, and analyzing notebook cells

In [ ]:
#| default_exp tools.editing

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
#| skip_showdoc
from __future__ import annotations
from typing import Any, Dict, List, Optional

import re
import json
import difflib
from pathlib import Path

from rich.panel import Panel
from mcp.server.fastmcp import FastMCP
from mcp.types import ToolAnnotations

from nbdev_mcp.utils.paths import (
    nbs_dir, settings_dict, resolve_selector, 
)
from nbdev_mcp.utils.nb import join_source, truncate
from nbdev_mcp.utils.md import split_markdown_into_cells
from nbdev_mcp.utils.rich import make_console, get_output, render_table

## Check Generated Files

In [ ]:
#| export
def check_if_generated(
    project: Optional[str] = None,
    file_path: str = ''
) -> Dict[str, Any]:
    """Check if a file is auto-generated by nbdev.
    
    USE THIS BEFORE EDITING ANY PYTHON FILE IN A NBDEV PROJECT!
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    file_path : str
        Path to the file to check.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'is_generated' bool and suggested action.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    s = settings_dict(p)
    lib_name = s.get('lib_name', '')
    is_in_lib = lib_name and (file_path.startswith(lib_name + '/') or file_path.startswith(lib_name + '\\'))
    is_py_file = file_path.endswith('.py')
    is_readme = file_path == 'README.md' or file_path.endswith('/README.md')
    
    if is_in_lib and is_py_file:
        nbs = nbs_dir(p)
        return {
            'ok': True,
            'is_generated': True,
            'file': file_path,
            'warning': '⚠️ This file is GENERATED by nbdev. DO NOT EDIT DIRECTLY!',
            'action': f"1. Use find_source_notebook(py_file='{file_path}') to find the source notebook\n2. Edit that notebook instead\n3. Run `nbdev_export` to regenerate this file",
            'pretty': f'⚠️ {file_path} is AUTO-GENERATED\nEdit the source notebook in {nbs}/ instead!'
        }
    
    if is_readme:
        return {
            'ok': True,
            'is_generated': True,
            'file': file_path,
            'warning': '⚠️ README.md is GENERATED from index.ipynb. DO NOT EDIT DIRECTLY!',
            'action': 'Edit nbs/index.ipynb instead, then run nbdev_readme()',
            'pretty': '⚠️ README.md is AUTO-GENERATED\nEdit nbs/index.ipynb instead!'
        }
    
    return {
        'ok': True,
        'is_generated': False,
        'file': file_path,
        'message': '✓ This file is safe to edit (not generated by nbdev)'
    }

In [ ]:
#| export
#| skip_showdoc
def find_source_notebook(
    project: Optional[str] = None,
    py_file: str = ''
) -> Dict[str, Any]:
    """Find which notebook generated a given Python file.
    
    CRITICAL: Use this to find the source before editing any .py file!
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    py_file : str
        Path to the Python file.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'notebook' path if found.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    if not py_file:
        return {'ok': False, 'error': 'py_file parameter is required'}
    
    s = settings_dict(p)
    lib_name = s.get('lib_name', '')
    nbs = nbs_dir(p)
    py_path = Path(py_file)
    
    if lib_name and py_file.startswith(lib_name):
        module_name = py_file.replace(lib_name + '/', '').replace(lib_name + '\\', '')
        module_name = Path(module_name).stem
    else:
        module_name = py_path.stem
    
    matching_notebooks = []
    if nbs.exists():
        for nb_path in nbs.rglob('*.ipynb'):
            if '.ipynb_checkpoints' in str(nb_path):
                continue
            try:
                nb_data = json.loads(nb_path.read_text(encoding='utf-8'))
                for cell in nb_data.get('cells', []):
                    if cell.get('cell_type') == 'code':
                        source = join_source(cell.get('source', []))
                        match = re.search(r'#\|\s*default_exp\s+(\w+)', source)
                        if match and match.group(1) == module_name:
                            matching_notebooks.append({
                                'notebook': str(nb_path.relative_to(p)),
                                'full_path': str(nb_path),
                                'module': module_name,
                                'confidence': 'high'
                            })
                            break
                        if nb_path.stem == module_name:
                            matching_notebooks.append({
                                'notebook': str(nb_path.relative_to(p)),
                                'full_path': str(nb_path),
                                'module': module_name,
                                'confidence': 'medium'
                            })
                            break
            except Exception:
                continue
    
    if not matching_notebooks:
        return {
            'ok': False,
            'error': f'Could not find source notebook for {py_file}',
            'hint': f"Expected notebook with '#| default_exp {module_name}' in {nbs}/"
        }
    
    best_match = sorted(matching_notebooks, key=lambda x: x['confidence'], reverse=True)[0]
    c = make_console()
    c.print(Panel(
        f"[bold green]✓[/] {py_file} is generated from:\n[cyan]{best_match['notebook']}[/]",
        title='Source Notebook Found'
    ))
    
    return {
        'ok': True,
        'py_file': py_file,
        'notebook': best_match['notebook'],
        'notebook_full_path': best_match['full_path'],
        'module': module_name,
        'message': f"✓ Source: {best_match['notebook']}",
        'pretty': get_output(c)
    }

## Analyze Exports

In [ ]:
#| export
#| skip_showdoc
def analyze_exports(
    project: Optional[str] = None,
    notebook: str = '',
    preview_length: int = 200
) -> Dict[str, Any]:
    """Analyze what a notebook exports (functions, classes, etc.).
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    notebook : str
        Notebook filename relative to nbs/ directory.
    preview_length : int
        Maximum chars for cell preview (default 200).
    
    Returns
    -------
    Dict[str, Any]
        Result with 'exports' list of exported symbols.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    nbs = nbs_dir(p)
    nb_path = (nbs / notebook).resolve() if '/' not in notebook else (p / notebook).resolve()
    
    if not nb_path.exists():
        return {'ok': False, 'error': f'Notebook not found: {notebook}'}
    
    try:
        nb_data = json.loads(nb_path.read_text(encoding='utf-8'))
    except Exception as e:
        return {'ok': False, 'error': f'Could not read notebook: {e}'}
    
    exports = []
    module_name = None
    
    for i, cell in enumerate(nb_data.get('cells', [])):
        if cell.get('cell_type') != 'code':
            continue
        source = join_source(cell.get('source', []))
        
        match = re.search(r'#\|\s*default_exp\s+(\w+)', source)
        if match:
            module_name = match.group(1)
        
        has_export_directive = bool(re.search(r'#\|\s*export\s*$', source, re.MULTILINE))
        has_exporti = bool(re.search(r'#\|\s*exporti', source))
        
        if has_export_directive or has_exporti:
            symbols = []
            for func_match in re.finditer(r'def\s+(\w+)\s*\(', source):
                symbols.append({'type': 'function', 'name': func_match.group(1)})
            for class_match in re.finditer(r'class\s+(\w+)\s*[\(:]', source):
                symbols.append({'type': 'class', 'name': class_match.group(1)})
            
            exports.append({
                'cell_index': i,
                'export_type': 'exporti' if has_exporti else 'export',
                'symbols': symbols,
                'preview': truncate(source, preview_length)
            })
    
    rows = []
    for exp in exports[:50]:
        symbols_str = ', '.join(s['name'] for s in exp['symbols']) or '(no named symbols)'
        rows.append([str(exp['cell_index']), exp['export_type'], symbols_str])
    if len(exports) > 50:
        rows.append(['...', '...', f'+ {len(exports) - 50} more'])
    pretty = render_table(f'Exports from {notebook}', ['Cell', 'Type', 'Symbols'], rows)
    
    return {
        'ok': True,
        'notebook': str(nb_path.relative_to(p)),
        'module': module_name,
        'export_count': len(exports),
        'exports': exports[:50],
        'total_exports': len(exports),
        'limited': len(exports) > 50,
        'pretty': pretty
    }

## Cell Reading and Editing

In [ ]:
#| export
#| skip_showdoc
def read_notebook_cell(
    project: Optional[str] = None,
    notebook: str = '',
    cell_index: Optional[int] = None,
    search: Optional[str] = None,
    max_results: int = 10,
    truncate_length: int = 2000
) -> Dict[str, Any]:
    """Read specific cells from a notebook by index or search.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    notebook : str
        Notebook filename.
    cell_index : int, optional
        Specific cell index to read.
    search : str, optional
        Search string to find in cells.
    max_results : int
        Maximum search results (default 10).
    truncate_length : int
        Max chars per cell (default 2000).
    
    Returns
    -------
    Dict[str, Any]
        Result with cell content or search results.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    nbs = nbs_dir(p)
    nb_path = (nbs / notebook).resolve() if '/' not in notebook else (p / notebook).resolve()
    
    if not nb_path.exists():
        return {'ok': False, 'error': f'Notebook not found: {notebook}'}
    
    try:
        nb_data = json.loads(nb_path.read_text(encoding='utf-8'))
    except Exception as e:
        return {'ok': False, 'error': f'Could not read notebook: {e}'}
    
    cells = nb_data.get('cells', [])
    
    if cell_index is not None:
        if cell_index < 0 or cell_index >= len(cells):
            return {'ok': False, 'error': f'Cell index {cell_index} out of range (0-{len(cells)-1})'}
        cell = cells[cell_index]
        source = join_source(cell.get('source', []))
        return {
            'ok': True,
            'notebook': str(nb_path.relative_to(p)),
            'cell_index': cell_index,
            'cell_type': cell.get('cell_type'),
            'source': truncate(source, truncate_length),
            'source_length': len(source),
            'truncated': len(source) > truncate_length
        }
    
    if search:
        matching_cells = []
        for i, cell in enumerate(cells):
            if len(matching_cells) >= max_results:
                break
            source = join_source(cell.get('source', []))
            if search.lower() in source.lower():
                matching_cells.append({
                    'cell_index': i,
                    'cell_type': cell.get('cell_type'),
                    'source': truncate(source, truncate_length),
                    'source_length': len(source),
                    'truncated': len(source) > truncate_length
                })
        
        total_matches = sum(
            1 for cell in cells
            if search.lower() in join_source(cell.get('source', [])).lower()
        )
        
        return {
            'ok': True,
            'notebook': str(nb_path.relative_to(p)),
            'search': search,
            'total_matches': total_matches,
            'returned': len(matching_cells),
            'limited': total_matches > max_results,
            'cells': matching_cells,
            'message': f'Showing {len(matching_cells)} of {total_matches} matches' if total_matches > max_results else f'Found {total_matches} matches'
        }
    
    return {'ok': False, 'error': 'Either cell_index or search must be provided'}

In [ ]:
#| export
#| skip_showdoc
def edit_notebook_cell(
    project: Optional[str] = None,
    notebook: str = '',
    cell_index: int = 0,
    new_source: str = ''
) -> Dict[str, Any]:
    """Edit a specific cell in a notebook.
    
    Run nbdev_export afterward to regenerate Python modules!
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    notebook : str
        Notebook filename.
    cell_index : int
        Index of cell to edit.
    new_source : str
        New source code for the cell.
    
    Returns
    -------
    Dict[str, Any]
        Result with confirmation and next steps.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    nbs = nbs_dir(p)
    nb_path = (nbs / notebook).resolve() if '/' not in notebook else (p / notebook).resolve()
    
    if not nb_path.exists():
        return {'ok': False, 'error': f'Notebook not found: {notebook}'}
    
    try:
        nb_data = json.loads(nb_path.read_text(encoding='utf-8'))
    except Exception as e:
        return {'ok': False, 'error': f'Could not read notebook: {e}'}
    
    cells = nb_data.get('cells', [])
    if cell_index < 0 or cell_index >= len(cells):
        return {'ok': False, 'error': f'Cell index {cell_index} out of range (0-{len(cells)-1})'}
    
    cells[cell_index]['source'] = new_source.splitlines(True)
    
    try:
        nb_path.write_text(json.dumps(nb_data, indent=2), encoding='utf-8')
    except Exception as e:
        return {'ok': False, 'error': f'Could not write notebook: {e}'}
    
    c = make_console()
    c.print(Panel(
        f'[green]✓[/] Updated cell {cell_index} in {notebook}\n\n[yellow]⚠ Next step: Run nbdev_export()[/]',
        title='Cell Edited'
    ))
    
    return {
        'ok': True,
        'notebook': str(nb_path.relative_to(p)),
        'cell_index': cell_index,
        'message': f'✓ Cell {cell_index} updated',
        'next_step': 'Run nbdev_export() to regenerate Python modules',
        'pretty': get_output(c)
    }

In [ ]:
#| export
#| skip_showdoc
def add_notebook_cell(
    project: Optional[str] = None,
    notebook: str = '',
    source: str = '',
    cell_type: str = 'code',
    position: str = 'auto',
    after_index: Optional[int] = None,
    export: bool = False,
    section_header: Optional[str] = None
) -> Dict[str, Any]:
    """Add a new cell to a notebook with smart positioning.
    
    Positions:
    - 'auto': Insert before the standard ending (## Next, ## Export)
    - 'import': Add to the imports section
    - 'start': Insert at beginning
    - 'end': Append to end
    - 'after': Insert after specific index
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    notebook : str
        Notebook filename.
    source : str
        Source code for the new cell.
    cell_type : str
        Cell type ('code' or 'markdown').
    position : str
        Where to insert: 'auto', 'import', 'start', 'end', or 'after'.
    after_index : int, optional
        Cell index to insert after (required if position='after').
    export : bool
        If True and cell_type='code', prepend ``#| export`` directive.
    section_header : str, optional
        If provided, add a markdown section header before the code cell.
    
    Returns
    -------
    Dict[str, Any]
        Result with inserted cell index.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    nbs = nbs_dir(p)
    nb_path = (nbs / notebook).resolve() if '/' not in notebook else (p / notebook).resolve()
    
    if not nb_path.exists():
        return {'ok': False, 'error': f'Notebook not found: {notebook}'}
    
    try:
        from nbdev_mcp.utils.nbformat import (
            read_notebook, write_notebook, 
            make_code_cell, make_markdown_cell, make_section_header,
            insert_cell, insert_imports, find_insertion_point,
            has_standard_ending
        )
        nb = read_notebook(nb_path)
    except Exception as e:
        return {'ok': False, 'error': f'Could not read notebook: {e}'}
    
    inserted_indices = []
    
    # Handle import position specially
    if position == 'import':
        idx = insert_imports(nb, source, merge=True)
        inserted_indices.append(idx)
    else:
        # Add section header if provided
        if section_header and cell_type == 'code':
            # Extract code names from source for backtick formatting
            code_names = []
            import ast
            try:
                tree = ast.parse(source)
                for node in ast.iter_child_nodes(tree):
                    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                        code_names.append(node.name)
                    elif isinstance(node, ast.ClassDef):
                        code_names.append(node.name)
            except SyntaxError:
                pass
            
            header_cell = make_section_header(section_header, code_names=code_names)
            idx = insert_cell(nb, header_cell, position=position, after_index=after_index)
            inserted_indices.append(idx)
            after_index = idx  # Insert code after header
            position = 'after'
        
        # Create the main cell
        if cell_type == 'code':
            cell = make_code_cell(source, export=export)
        else:
            cell = make_markdown_cell(source)
        
        idx = insert_cell(nb, cell, position=position, after_index=after_index)
        inserted_indices.append(idx)
    
    try:
        write_notebook(nb, nb_path)
    except Exception as e:
        return {'ok': False, 'error': f'Could not write notebook: {e}'}
    
    insert_idx = inserted_indices[-1]
    
    return {
        'ok': True,
        'notebook': str(nb_path.relative_to(p)),
        'inserted_at': insert_idx,
        'inserted_indices': inserted_indices,
        'has_standard_ending': has_standard_ending(nb),
        'message': f'✓ Added {cell_type} cell at index {insert_idx}',
        'next_step': 'Run nbdev_export() if this is an export cell'
    }


In [ ]:
#| export
#| skip_showdoc
def split_markdown_cells(markdown: str) -> Dict[str, Any]:
    """Split a markdown blob into notebook-friendly markdown cells.
    
    Rules:
    - Each heading line (``#``, ``##``, ...) is its own cell
    - Text after a heading becomes a separate cell
    - Reference link definitions are propagated to cells that use them
    
    Parameters
    ----------
    markdown : str
        Full markdown document text.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'cells' list ready for notebook insertion.
    """
    cells = split_markdown_into_cells(markdown)
    preview = [{'index': i, 'source': c['source'][:160]} for i, c in enumerate(cells[:5])]
    
    return {
        'ok': True,
        'count': len(cells),
        'cells': cells,
        'preview': preview,
        'message': f'Split into {len(cells)} markdown cells'
    }

## Notebook Diff

In [ ]:
#| export
#| skip_showdoc
def notebook_diff(
    project: Optional[str] = None,
    notebook: str = '',
    other: Optional[str] = None,
    cell_index: Optional[int] = None,
    context_lines: int = 3
) -> Dict[str, Any]:
    """Show diff between notebook versions or cells.
    
    Can compare:
    - Two different notebooks (notebook vs other)
    - A specific cell against git HEAD
    - Entire notebook against git HEAD (if other not provided)
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    notebook : str
        Primary notebook filename relative to nbs/.
    other : str, optional
        Second notebook to compare against. If not provided, compares against git HEAD.
    cell_index : int, optional
        Specific cell to diff. If not provided, diffs entire notebook.
    context_lines : int
        Number of context lines around changes (default: 3).
    
    Returns
    -------
    Dict[str, Any]
        Result with 'diff' text and 'changes' summary.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    nbs = nbs_dir(p)
    nb_path = (nbs / notebook).resolve() if '/' not in notebook else (p / notebook).resolve()
    
    if not nb_path.exists():
        return {'ok': False, 'error': f'Notebook not found: {notebook}'}
    
    try:
        nb_data = json.loads(nb_path.read_text(encoding='utf-8'))
    except Exception as e:
        return {'ok': False, 'error': f'Could not read notebook: {e}'}
    
    # Get comparison data
    if other:
        # Compare two notebooks
        other_path = (nbs / other).resolve() if '/' not in other else (p / other).resolve()
        if not other_path.exists():
            return {'ok': False, 'error': f'Other notebook not found: {other}'}
        try:
            other_data = json.loads(other_path.read_text(encoding='utf-8'))
        except Exception as e:
            return {'ok': False, 'error': f'Could not read other notebook: {e}'}
        compare_label = other
    else:
        # Compare against git HEAD
        import subprocess
        try:
            rel_path = nb_path.relative_to(p)
            result = subprocess.run(
                ['git', 'show', f'HEAD:{rel_path}'],
                capture_output=True, text=True, cwd=p
            )
            if result.returncode != 0:
                return {'ok': False, 'error': f'Could not get git HEAD version: {result.stderr}'}
            other_data = json.loads(result.stdout)
            compare_label = 'HEAD'
        except FileNotFoundError:
            return {'ok': False, 'error': 'git not available'}
        except json.JSONDecodeError:
            return {'ok': False, 'error': 'Could not parse git HEAD version'}
    
    # Extract cells to compare
    if cell_index is not None:
        cells = nb_data.get('cells', [])
        other_cells = other_data.get('cells', [])
        
        if cell_index >= len(cells):
            return {'ok': False, 'error': f'Cell {cell_index} not found in {notebook}'}
        
        current_src = join_source(cells[cell_index].get('source', [])).splitlines(keepends=True)
        
        if cell_index < len(other_cells):
            other_src = join_source(other_cells[cell_index].get('source', [])).splitlines(keepends=True)
        else:
            other_src = []
        
        diff = list(difflib.unified_diff(
            other_src, current_src,
            fromfile=f'{compare_label}:cell[{cell_index}]',
            tofile=f'{notebook}:cell[{cell_index}]',
            n=context_lines
        ))
    else:
        # Diff entire notebook cell by cell
        cells = nb_data.get('cells', [])
        other_cells = other_data.get('cells', [])
        
        diff = []
        max_cells = max(len(cells), len(other_cells))
        
        for i in range(max_cells):
            current_src = join_source(cells[i].get('source', [])).splitlines(keepends=True) if i < len(cells) else []
            other_src = join_source(other_cells[i].get('source', [])).splitlines(keepends=True) if i < len(other_cells) else []
            
            cell_diff = list(difflib.unified_diff(
                other_src, current_src,
                fromfile=f'{compare_label}:cell[{i}]',
                tofile=f'{notebook}:cell[{i}]',
                n=context_lines
            ))
            if cell_diff:
                diff.extend(cell_diff)
                diff.append('\n')
    
    diff_text = ''.join(diff)
    
    # Count changes
    additions = sum(1 for line in diff if line.startswith('+') and not line.startswith('+++'))
    deletions = sum(1 for line in diff if line.startswith('-') and not line.startswith('---'))
    
    c = make_console()
    if diff_text.strip():
        c.print(Panel(diff_text[:5000] + ('...' if len(diff_text) > 5000 else ''), title=f'Diff: {notebook} vs {compare_label}'))
    else:
        c.print(Panel('No differences found', title='notebook_diff'))
    
    return {
        'ok': True,
        'notebook': str(nb_path.relative_to(p)),
        'compared_to': compare_label,
        'diff': diff_text,
        'additions': additions,
        'deletions': deletions,
        'has_changes': bool(diff_text.strip()),
        'pretty': get_output(c)
    }


In [ ]:
#| export
#| skip_showdoc
import subprocess
import tempfile

def execute_cell(
    project: Optional[str] = None,
    notebook: str = '',
    cell_index: int = 0,
    timeout: int = 60
) -> Dict[str, Any]:
    """Execute a specific cell from a notebook and return its output.
    
    Uses execnb to execute the cell in isolation. Code cells are executed
    and their outputs captured. Markdown cells are returned as-is.
    
    Parameters
    ----------
    project : str, optional
        Path or alias for nbdev project.
    notebook : str
        Path to the notebook (relative to nbs/).
    cell_index : int
        The 0-based index of the cell to execute.
    timeout : int, optional
        Execution timeout in seconds (default 60).
    
    Returns
    -------
    Dict[str, Any]
        Result dict with:
        - ok: bool
        - cell_type: 'code' or 'markdown'
        - source: cell source code
        - outputs: list of output dicts (for code cells)
        - stdout: captured stdout
        - stderr: captured stderr
        - execution_count: int
        - error: str if execution failed
    """
    from nbdev_mcp.utils.paths import nbs_dir, resolve_selector, read_nb, write_nb
    from nbdev_mcp.utils.nb import join_source
    
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    # Find the notebook
    nbs = nbs_dir(p)
    nb_path = nbs / notebook if not Path(notebook).is_absolute() else Path(notebook)
    if not nb_path.exists():
        nb_path = p / notebook
    if not nb_path.exists():
        return {'ok': False, 'error': f'Notebook not found: {notebook}'}
    
    # Read the notebook
    try:
        data = read_nb(nb_path)
    except Exception as e:
        return {'ok': False, 'error': f'Failed to read notebook: {e}'}
    
    cells = data.get('cells', [])
    if cell_index < 0 or cell_index >= len(cells):
        return {'ok': False, 'error': f'Cell index {cell_index} out of range (0-{len(cells)-1})'}
    
    cell = cells[cell_index]
    cell_type = cell.get('cell_type', 'code')
    source = join_source(cell.get('source', []))
    
    # Markdown cells don't need execution
    if cell_type != 'code':
        return {
            'ok': True,
            'cell_type': cell_type,
            'cell_index': cell_index,
            'source': source,
            'message': 'Markdown cells cannot be executed'
        }
    
    # Execute the code cell using a subprocess with execnb
    # Create a temporary notebook with just this cell
    try:
        temp_nb = {
            'cells': [cell],
            'metadata': data.get('metadata', {}),
            'nbformat': data.get('nbformat', 4),
            'nbformat_minor': data.get('nbformat_minor', 5)
        }
        
        with tempfile.NamedTemporaryFile(mode='w', suffix='.ipynb', delete=False) as f:
            import json
            json.dump(temp_nb, f)
            temp_path = f.name
        
        # Execute using execnb
        result = subprocess.run(
            ['python', '-c', f'''
import json
from pathlib import Path
from execnb.nbio import read_nb, write_nb
from execnb.core import CaptureShell

nb = read_nb("{temp_path}")
shell = CaptureShell()
try:
    shell.run(nb)
except Exception as e:
    nb["cells"][0]["outputs"] = [{{"output_type": "error", "ename": type(e).__name__, "evalue": str(e), "traceback": []}}]
write_nb(nb, "{temp_path}")
'''],
            capture_output=True,
            text=True,
            timeout=timeout,
            cwd=str(p)
        )
        
        # Read back the executed notebook
        with open(temp_path, 'r') as f:
            executed_nb = json.load(f)
        
        executed_cell = executed_nb['cells'][0]
        outputs = executed_cell.get('outputs', [])
        
        # Extract stdout/stderr from outputs
        stdout_parts = []
        stderr_parts = []
        for out in outputs:
            if out.get('output_type') == 'stream':
                if out.get('name') == 'stdout':
                    stdout_parts.append(out.get('text', ''))
                elif out.get('name') == 'stderr':
                    stderr_parts.append(out.get('text', ''))
        
        # Check for errors
        error_output = None
        for out in outputs:
            if out.get('output_type') == 'error':
                error_output = {
                    'ename': out.get('ename', 'Error'),
                    'evalue': out.get('evalue', ''),
                    'traceback': out.get('traceback', [])
                }
                break
        
        # Clean up
        Path(temp_path).unlink(missing_ok=True)
        
        return {
            'ok': error_output is None,
            'cell_type': 'code',
            'cell_index': cell_index,
            'source': source,
            'outputs': outputs,
            'stdout': ''.join(stdout_parts),
            'stderr': ''.join(stderr_parts) + result.stderr,
            'execution_count': executed_cell.get('execution_count'),
            'error': error_output
        }
        
    except subprocess.TimeoutExpired:
        Path(temp_path).unlink(missing_ok=True)
        return {'ok': False, 'error': f'Execution timed out after {timeout}s'}
    except Exception as e:
        return {'ok': False, 'error': f'Execution failed: {e}'}

## MCP Registration

In [ ]:
#| export
#| skip_showdoc
# Tool annotation definitions for notebook editing tools
EDITING_TOOL_ANNOTATIONS = {
    'check_if_generated': ToolAnnotations(
        title="Check If Generated",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'find_source_notebook': ToolAnnotations(
        title="Find Source Notebook",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'analyze_exports': ToolAnnotations(
        title="Analyze Exports",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'read_notebook_cell': ToolAnnotations(
        title="Read Notebook Cell",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'edit_notebook_cell': ToolAnnotations(
        title="Edit Notebook Cell",
        readOnlyHint=False,
        idempotentHint=False,
        openWorldHint=False
    ),
    'add_notebook_cell': ToolAnnotations(
        title="Add Notebook Cell",
        readOnlyHint=False,
        idempotentHint=False,
        openWorldHint=False
    ),
    'split_markdown_cells': ToolAnnotations(
        title="Split Markdown Cells",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'notebook_diff': ToolAnnotations(
        title="Notebook Diff",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'execute_cell': ToolAnnotations(
        title="Execute Cell",
        readOnlyHint=True,  # Doesn't modify the original notebook
        idempotentHint=False,  # Execution may have side effects
        openWorldHint=False
    ),
}

def add_notebook_editing_tools(mcp: FastMCP) -> None:
    """Attach notebook editing tools to the MCP server with annotations.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance.
    """
    mcp.add_tool(check_if_generated, annotations=EDITING_TOOL_ANNOTATIONS['check_if_generated'])
    mcp.add_tool(find_source_notebook, annotations=EDITING_TOOL_ANNOTATIONS['find_source_notebook'])
    mcp.add_tool(analyze_exports, annotations=EDITING_TOOL_ANNOTATIONS['analyze_exports'])
    mcp.add_tool(read_notebook_cell, annotations=EDITING_TOOL_ANNOTATIONS['read_notebook_cell'])
    mcp.add_tool(edit_notebook_cell, annotations=EDITING_TOOL_ANNOTATIONS['edit_notebook_cell'])
    mcp.add_tool(add_notebook_cell, annotations=EDITING_TOOL_ANNOTATIONS['add_notebook_cell'])
    mcp.add_tool(split_markdown_cells, annotations=EDITING_TOOL_ANNOTATIONS['split_markdown_cells'])
    mcp.add_tool(notebook_diff, annotations=EDITING_TOOL_ANNOTATIONS['notebook_diff'])
    mcp.add_tool(execute_cell, annotations=EDITING_TOOL_ANNOTATIONS['execute_cell'])

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()